# Path A, Step 2, P5 — target 8: reversal rate (Oct 26, 2026)

Rate = reversals / human player-minutes, pooled over the runs of a cell (topology x dataset); a round lasts until `fraught.round_end`. Reversals: `detect_events` with m = 2 on every human player. 95% CI: group bootstrap (log files, 10,000 resamples, seed 573), the P2c method. Function: `targets.reversal_rate_table` (`src/slider/targets.py`).

- **a) P2c set:** `load_runs("behavior")` (train groups, bots and incomplete runs kept), CC through `lab_log` (tick 0.1 s). Must reproduce Ch. A4 (the P2c table).
- **b) Target 8:** train + val groups, no bots, no incomplete runs; CC through `lab_log` (tick 0.1 s).
- **c) Check:** the runs of (b), CC on its raw rows.

Lab data (EL, CL) are used as logged in all three. Saved: `results/tables/target8_reversal_rate.csv`.

In [1]:
import numpy as np
import pandas as pd

from slider import config
from slider import slider_logs as sl
from slider import targets
from slider.labclock import CELLS

pd.set_option("display.max_columns", 30)

runs_a, table_a = sl.load_runs("behavior")
runs_b, table_b = sl.load_runs("behavior", roles={"train", "val"}, keep_bot=False, keep_incomplete=False)

a = targets.reversal_rate_table(runs_a)
b = targets.reversal_rate_table(runs_b)
c = targets.reversal_rate_table(runs_b, lab_clock_cc=False)

## Reproduction check: (a) against Ch. A4

Ch. A4 values: the P2c table (`p2c_event_counts.ipynb`, section a), 3 decimals.

In [2]:
a4 = pd.DataFrame([("Hierarchy", "EL", 11, 33, 208.208, 4.097, 3.172, 5.279),
                   ("Hierarchy", "CL", 11, 33, 272.446, 3.366, 2.442, 4.419),
                   ("Hierarchy", "CC", 17, 50, 279.198, 3.091, 2.681, 3.528),
                   ("Full", "CL", 11, 33, 375.662, 3.956, 2.964, 5.037),
                   ("Full", "CC", 17, 48, 284.875, 4.907, 4.402, 5.366),
                   ("Star", "CC", 17, 47, 189.735, 3.067, 2.523, 3.604)],
                  columns=["topology", "dataset", "groups", "runs", "player_min", "rate", "ci_lo", "ci_hi"])
cols = ["groups", "runs", "player_min", "rate", "ci_lo", "ci_hi"]
mine = a.set_index(["topology", "dataset"])[cols].round(3)
a4 = a4.set_index(["topology", "dataset"])
check = pd.concat({"a": mine, "Ch. A4": a4}, axis=1)
check["match"] = np.isclose(mine, a4, rtol=0, atol=1e-9).all(axis=1)
check

a                                      Ch. A4       \
                  groups runs player_min   rate  ci_lo  ci_hi groups runs   
topology  dataset                                                           
Hierarchy EL          11   33    208.208  4.097  3.172  5.279     11   33   
          CL          11   33    272.446  3.366  2.442  4.419     11   33   
          CC          17   50    279.198  3.091  2.681  3.528     17   50   
Full      CL          11   33    375.662  3.956  2.964  5.037     11   33   
          CC          17   48    284.875  4.907  4.402  5.366     17   48   
Star      CC          17   47    189.735  3.067  2.523  3.604     17   47   

                                                  match  
                  player_min   rate  ci_lo  ci_hi        
topology  dataset                                        
Hierarchy EL         208.208  4.097  3.172  5.279  True  
          CL         272.446  3.366  2.442  4.419  True  
          CC         279.198  3.091  2.681  3.528  True  
Full      CL         375.662  3.956  2.964  5.037  True  
          CC         284.875  4.907  4.402  5.366  True  
Star      CC         189.735  3.067  2.523  3.604  True

## Target 8: a, b and c side by side

In [3]:
cols = ["groups", "runs", "reversals", "player_min", "rate", "ci_lo", "ci_hi"]
sets = {"a) P2c set": a, "b) target 8": b, "c) raw CC": c}
side = pd.concat({name: d.set_index(["topology", "dataset"])[cols] for name, d in sets.items()}, axis=1)
side.round(3)

a) P2c set                                                 \
                      groups runs reversals player_min   rate  ci_lo  ci_hi   
topology  dataset                                                             
Hierarchy EL              11   33       853    208.208  4.097  3.172  5.279   
          CL              11   33       917    272.446  3.366  2.442  4.419   
          CC              17   50       863    279.198  3.091  2.681  3.528   
Full      CL              11   33      1486    375.662  3.956  2.964  5.037   
          CC              17   48      1398    284.875  4.907  4.402  5.366   
Star      CC              17   47       582    189.735  3.067  2.523  3.604   

                  b) target 8                                                 \
                       groups runs reversals player_min   rate  ci_lo  ci_hi   
topology  dataset                                                              
Hierarchy EL               17   51      1610    438.834  3.669  2.917  4.604   
          CL               18   53      1229    424.586  2.895  2.163  3.702   
          CC               16   40       940    260.548  3.608  2.865  4.608   
Full      CL               18   54      2864    745.209  3.843  3.063  4.676   
          CC               16   36      1245    241.236  5.161  4.573  5.795   
Star      CC               17   39       622    181.706  3.423  2.789  4.037   

                  c) raw CC                                                 
                     groups runs reversals player_min   rate  ci_lo  ci_hi  
topology  dataset                                                           
Hierarchy EL             17   51      1610    438.834  3.669  2.917  4.604  
          CL             18   53      1229    424.586  2.895  2.163  3.702  
          CC             16   40       942    260.548  3.615  2.865  4.631  
Full      CL             18   54      2864    745.209  3.843  3.063  4.676  
          CC             16   36      1247    241.236  5.169  4.574  5.813  
Star      CC             17   39       622    181.706  3.423  2.789  4.037

## Runs from (a) to (b), by cell

Dropped = runs of (a) that are not in (b) (a run can have a bot and be incomplete). Added = val runs.

In [4]:
cell = ["topology", "dataset"]
in_b = set(zip(table_b["file"], table_b["wave"]))
dropped = table_a[[run not in in_b for run in zip(table_a["file"], table_a["wave"])]]
moves = pd.DataFrame({"runs a": table_a.groupby(cell).size(),
                      "dropped": dropped.groupby(cell).size(),
                      "dropped, bot": dropped[dropped["has_bot"]].groupby(cell).size(),
                      "dropped, incomplete": dropped[dropped["end"] == "incomplete"].groupby(cell).size(),
                      "added (val)": table_b[table_b["role"] == "val"].groupby(cell).size(),
                      "runs b": table_b.groupby(cell).size()})
moves = moves.reindex(CELLS).fillna(0).astype(int)
assert (moves["runs a"] - moves["dropped"] + moves["added (val)"] == moves["runs b"]).all()
moves

runs a  dropped  dropped, bot  dropped, incomplete  \
topology  dataset                                                       
Hierarchy EL           33        0             0                    0   
          CL           33        0             0                    0   
          CC           50       16            14                    2   
Full      CL           33        0             0                    0   
          CC           48       17            14                    5   
Star      CC           47       14            14                    2   

                   added (val)  runs b  
topology  dataset                       
Hierarchy EL                18      51  
          CL                20      53  
          CC                 6      40  
Full      CL                21      54  
          CC                 5      36  
Star      CC                 6      39

## Save

In [5]:
names = {"a) P2c set": "a_p2c_set", "b) target 8": "b_target8", "c) raw CC": "c_target8_raw_cc"}
out = pd.concat([d.assign(set=names[name]) for name, d in sets.items()])
out = out[["set"] + list(a.columns)]
out.to_csv(config.TARGET8, index=False)
out.round(3)

,set,topology,dataset,groups,runs,reversals,player_min,rate,ci_lo,ci_hi
0,a_p2c_set,Hierarchy,EL,11,33,853,208.208,4.097,3.172,5.279
1,a_p2c_set,Hierarchy,CL,11,33,917,272.446,3.366,2.442,4.419
2,a_p2c_set,Hierarchy,CC,17,50,863,279.198,3.091,2.681,3.528
3,a_p2c_set,Full,CL,11,33,1486,375.662,3.956,2.964,5.037
4,a_p2c_set,Full,CC,17,48,1398,284.875,4.907,4.402,5.366
5,a_p2c_set,Star,CC,17,47,582,189.735,3.067,2.523,3.604
0,b_target8,Hierarchy,EL,17,51,1610,438.834,3.669,2.917,4.604
1,b_target8,Hierarchy,CL,18,53,1229,424.586,2.895,2.163,3.702
2,b_target8,Hierarchy,CC,16,40,940,260.548,3.608,2.865,4.608
3,b_target8,Full,CL,18,54,2864,745.209,3.843,3.063,4.676
